# Olist：履约体验与复购基线

先按 [README](../README.md) 导入数据。Notebook 使用已有的 MySQL 库和仓库的汇总查询。需要 Pandas 和 Jupyter（可运行 python -m pip install pandas notebook），连接密码在运行时隐藏输入。

In [ ]:
from getpass import getpass
from pathlib import Path
import sys

import pandas as pd
import pymysql
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "scripts" / "run_analysis.py").is_file():
    raise FileNotFoundError("请从仓库根目录或 notebooks 目录启动 Jupyter")
sys.path.insert(0, str(root / "scripts"))
from run_analysis import SQL, STAGES, iter_results

user = input("MySQL 用户名 [root]: ").strip() or "root"
port = int(input("MySQL 端口 [3306]: ").strip() or "3306")
if "connection" in globals() and connection.open:
    connection.close()
connection = pymysql.connect(
    host="127.0.0.1", port=port, user=user,
    password=getpass("MySQL 密码："), charset="utf8mb4",
)

def results(stage):
    for i, (columns, rows) in enumerate(iter_results(connection, SQL / STAGES[stage]), 1):
        print(f"{stage} {i}")
        display(pd.DataFrame(rows, columns=columns))


## 1. 质量检查
检查入库行数、订单视图唯一性，以及商品、运费和支付三项金额差额。

In [ ]:
results("quality")

## 2. 基础指标和履约诊断
基础分析中的 A2 和 A3 分别为 2018 年上半年和圣保罗州的筛选案例。履约诊断包含全样本主窗口基线和里约热内卢州的分层分析。

In [ ]:
results("baseline")
results("fulfillment")

## 3. 完整观察期复购
使用数据内首购作为队列起点。合并同日购买，并为最长窗口留足 90 天。

In [ ]:
results("repeat")

## 4. 评论覆盖与固定抽样规模
只展示分组计数。人工试标的规则与问题汇总见 [报告](../docs/analysis.md#评论核验)。

In [ ]:
results("reviews")

In [ ]:
connection.close()